In [1]:
import pandas as pd
import polars as pl
import seaborn as sb
import lets_plot as lp
import sklearn
from sklearn.preprocessing import LabelEncoder
import numpy as np
pl.Config.set_tbl_cols(50)  # show up to 50 columns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score,
                             precision_recall_curve, auc, f1_score, make_scorer)
import xgboost as xgb
from xgboost import XGBClassifier


In [2]:
import xgboost as xgb
print(xgb.build_info())
import xgboost as xgb
print(f"XGBoost version: {xgb.__version__}")

{'BUILTIN_PREFETCH_PRESENT': True, 'CUDA_VERSION': [12, 9], 'DEBUG': False, 'GCC_VERSION': [10, 3, 1], 'GLIBC_VERSION': [2, 28], 'MM_PREFETCH_PRESENT': True, 'NCCL_VERSION': [2, 29, 2], 'THRUST_VERSION': [2, 8, 2], 'USE_CUDA': True, 'USE_DLOPEN_NCCL': True, 'USE_FEDERATED': True, 'USE_NCCL': True, 'USE_NVCOMP': False, 'USE_OPENMP': True, 'USE_RMM': False, 'libxgboost': '/home/treyt/Documents/BYUI/2026/machine-learning/uv-final-project/.venv/lib/python3.13/site-packages/xgboost/lib/libxgboost.so'}
XGBoost version: 3.2.0


In [3]:
DATA_DIR = "/home/treyt/Documents/BYUI/2026/machine-learning/uv-final-project/data/"
TRAIN_FRAC = .7
houses = [
 "csh101", 
 "csh102", 
 "csh103", 
 "csh104", 
 "csh105", 
 "csh106", 
 "csh107", 
 "csh108", 
 "csh109", 
 "csh110", 
 "csh111", 
 "csh112", 
 "csh113", 
 "csh114", 
 "csh115", 
 "csh116", 
 "csh117", 
 "csh118", 
 "csh119", 
 "csh120", 
 "csh121", 
 "csh122", 
 "csh123", 
 "csh124", 
 "csh125", 
 "csh126", 
 "csh127", 
 "csh128", 
 "csh129", 
 "csh130", 
]
print("Hello World!")


Hello World!


In [4]:
df_list = [ pl.scan_csv( f"{DATA_DIR}raw/{house_name}/{house_name}.ann.features.csv") for house_name in houses[:30] ]

In [5]:
!ls

01_casas_data_exploration.ipynb  exploratory.ipynb  exploratory.md


In [22]:
#df_all = pl.concat(pl.collect_all(df_list), how='diagonal')
df_sampled = [df.collect().sample(fraction=.8) for df in df_list]
df_sampled = pl.concat(
    df_sampled,
    how='diagonal'
)
le = LabelEncoder()


In [ ]:
X = df_sampled.drop('activity')
y = le.fit_transform(df_sampled['activity'])
print(pl.Series(df_sampled['activity']).value_counts().sort(by='count', descending=False).filter((pl.col('count') < 1000) & pl.col('activity').) )



shape: (8, 2)
┌─────────────────────┬───────┐
│ activity            ┆ count │
│ ---                 ┆ ---   │
│ str                 ┆ u32   │
╞═════════════════════╪═══════╡
│ r2.Dress            ┆ 1     │
│ r2.Eat_Breakfast    ┆ 1     │
│ r1.Cook_Breakfast   ┆ 2     │
│ r2.Personal_Hygiene ┆ 2     │
│ r1.Sleep            ┆ 4     │
│ Exercise            ┆ 72    │
│ Wake_Up             ┆ 296   │
│ Go_To_Sleep         ┆ 327   │
└─────────────────────┴───────┘


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(df_sampled.drop('activity'), le.fit_transform(df_sampled['activity']), test_size=0.33, random_state=42)

In [ ]:
X = df_sampled.drop('activity')
y = le.fit_transform(df_sampled['activity'])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
clf = XGBClassifier(
    random_state=42,
    eval_metric='mlogloss',
    n_estimators=400,        # start a bit higher
    learning_rate=0.09,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    n_jobs=-3,
    verbosity=3  # Shows training progress
)

In [ ]:
f1 = cross_val_score(clf, X, y,
                     cv=cv,
                     scoring=make_scorer(f1_score, average='weighted'),
                     n_jobs=-1)
print("5-fold weighted-F1: %.3f ± %.3f" % (f1.mean(), f1.std()))

/home/treyt/Documents/BYUI/2026/machine-learning/uv-final-project/.venv/lib/python3.13/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


[20:58:33] INFO: /__w/xgboost/xgboost/src/data/iterative_dmatrix.cc:56: Finished constructing the `IterativeDMatrix`: (1116511, 36, 40194396).
[20:58:35] INFO: /__w/xgboost/xgboost/src/data/iterative_dmatrix.cc:56: Finished constructing the `IterativeDMatrix`: (1116511, 36, 40194396).
[20:58:37] INFO: /__w/xgboost/xgboost/src/data/iterative_dmatrix.cc:56: Finished constructing the `IterativeDMatrix`: (1116511, 36, 40194396).
[20:58:40] INFO: /__w/xgboost/xgboost/src/data/iterative_dmatrix.cc:56: Finished constructing the `IterativeDMatrix`: (1116512, 36, 40194432).


In [ ]:
# Train the model
model = xgb.XGBClassifier( random_state=42, verbose=2)
model.fit(X_train, y_train)

/home/treyt/Documents/BYUI/2026/machine-learning/uv-final-project/.venv/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [23:11:40] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
/home/treyt/Documents/BYUI/2026/machine-learning/uv-final-project/.venv/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [23:11:40] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)
/home/treyt/Documents/BYUI/2026/machine-learning/uv-final-project/.venv/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [23:11:40] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "verbose" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


KeyboardInterrupt: 

In [ ]:
# predictions
from sklearn.metrics import f1_score


y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)  # shape (n_samples, n_classes)

# basic metrics
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 score:", f1_score(y_test, y_pred, average='weighted'))
#print(classification_report(y_test, y_pred, target_names=le.classes_, labels=))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

# ROC AUC (multiclass: one-vs-rest)
if y_proba.shape[1] == 2:
    roc = roc_auc_score(y_test, y_proba[:,1])
else:
    roc = roc_auc_score(y_test, y_proba, multi_class='ovr')
print("ROC AUC:", roc)

# Precision-Recall AUC for positive class (example for class 1)
precision, recall, _ = precision_recall_curve(y_test == 1, y_proba[:, 1])
pr_auc = auc(recall, precision)
print("PR AUC for class 1:", pr_auc)